In [24]:
xl = "../Data/DebtPenny_20251105_20261104.csv"   # two dots, not three
d = pd.read_csv(xl)
rec = d["Record Date"].astype(str)
d["date"] = pd.to_datetime(rec, format="%Y-%m-%d" if rec.str.match(r"\d{4}-").all() else "%d/%m/%Y")
d["debt"] = d["Total Public Debt Outstanding"].astype(float) / 1e12
debt = d[["date", "debt"]].sort_values("date").reset_index(drop=True)
assert debt.date.max() == pd.Timestamp("2026-11-04"), debt.date.max()   # catches any day/month mix-up
debt["series"] = "Actual"
last = debt.iloc[-1]
print(debt.tail(1))                            # should read 2026-11-04, 40.55

          date      debt  series
250 2026-11-04  40.55023  Actual


In [4]:
import pandas as pd, altair as alt
from ecostyles import EcoStyles            # adjust import
styles = EcoStyles(); styles.register_and_enable_theme()

LIMIT = 41.1                                   # statutory debt limit, $tn
ELECTION = pd.Timestamp("2026-11-03")          # midterm election day
NEUTRAL = "#676A86"                            # ecostyles axis grey, for the election marker only

xl = "../Data/DebtPenny_20251105_20261104.csv"
d = pd.read_csv(xl)
rec = d["Record Date"].astype(str)
d["date"] = pd.to_datetime(rec, format="%Y-%m-%d" if rec.str.match(r"\d{4}-").all() else "%d/%m/%Y")
d["debt"] = d["Total Public Debt Outstanding"].astype(float) / 1e12
debt = d[["date", "debt"]].sort_values("date").reset_index(drop=True)
assert debt.date.max() == pd.Timestamp("2026-11-04"), debt.date.max()   # catches any day/month mix-up
debt["series"] = "Actual"
print(debt.tail(1))                            # should read 2026-11-04, 40.55

# colour comes from the theme palette (teal, then red) via the series field -- no hard-coded hex
col = alt.Color("series:N", scale=alt.Scale(domain=["Actual", "Limit"]), legend=None)
END = "2027-02-15"
XS = alt.Scale(domain=["2025-11-01", END]); YS = alt.Scale(domain=[38, 41.6], nice=False)
X = alt.X("date:T", title=None, scale=XS, axis=alt.Axis(format="%b %Y", tickCount=5))
Y = alt.Y("debt:Q", title=None, scale=YS, axis=alt.Axis(values=[38, 39, 40, 41], labelExpr="'$' + datum.value + 'tn'"))

hist = alt.Chart(debt).mark_line(strokeWidth=1.8).encode(X, Y, color=col)
limit = alt.Chart(pd.DataFrame({"debt": [LIMIT], "series": ["Limit"]})).mark_rule(strokeWidth=1.8).encode(Y, color=col)
election = alt.Chart(pd.DataFrame({"date": [ELECTION]})).mark_rule(color=NEUTRAL, strokeDash=[4, 3], strokeWidth=1.5).encode(alt.X("date:T", scale=XS))

def text(x, y, s, align="left", dx=0, dy=0, series=None):
    dd = pd.DataFrame({"date": [pd.Timestamp(x)], "debt": [y], "series": [series or "Limit"]})
    m = alt.Chart(dd).mark_text(text=s, align=align, dx=dx, dy=dy, fontSize=14)
    enc = dict(x=alt.X("date:T", scale=XS), y=alt.Y("debt:Q", scale=YS))
    return m.encode(**enc, color=col) if series else m.encode(**enc, color=alt.value(NEUTRAL))

labels = (text("2025-11-08", 39.25, "Total public debt", series="Actual")
          + text(END, LIMIT, "Debt limit", align="right", dy=-10, series="Limit")
          + text(ELECTION, 41.45, "Midterm election day", align="right", dx=-6))

chart = (election + limit + hist + labels).properties(width=700, height=270)   # title/subtitle live in the text

styles.save(chart, path=".", name="debt_limit_midterms", svg=True)
chart

          date      debt  series
250 2026-11-04  40.55023  Actual


alt.LayerChart(...)

In [5]:
import pandas as pd, altair as alt
from ecostyles import EcoStyles            # adjust import
styles = EcoStyles(); styles.register_and_enable_theme()

LIMIT = 41.1                                   # statutory debt limit, $tn
ELECTION = pd.Timestamp("2026-11-03")          # midterm election day
FS = None                                      # set e.g. 11 to shrink ALL chart text (axes + labels); None = theme default
NEUTRAL = "#676A86"                            # ecostyles axis grey, for the election marker only

xl = "../Data/DebtPenny_20251105_20261104.csv"
d = pd.read_csv(xl)
rec = d["Record Date"].astype(str)
d["date"] = pd.to_datetime(rec, format="%Y-%m-%d" if rec.str.match(r"\d{4}-").all() else "%d/%m/%Y")
d["debt"] = d["Total Public Debt Outstanding"].astype(float) / 1e12
debt = d[["date", "debt"]].sort_values("date").reset_index(drop=True)
assert debt.date.max() == pd.Timestamp("2026-11-04"), debt.date.max()   # catches any day/month mix-up
debt["series"] = "Actual"
print(debt.tail(1))                            # should read 2026-11-04, 40.55

# colour comes from the theme palette (teal, then red) via the series field -- no hard-coded hex
col = alt.Color("series:N", scale=alt.Scale(domain=["Actual", "Limit"]), legend=None)
END = "2027-02-15"
XS = alt.Scale(domain=["2025-11-01", END]); YS = alt.Scale(domain=[38, 41.6], nice=False)
X = alt.X("date:T", title=None, scale=XS, axis=alt.Axis(format="%b %Y", tickCount=5))
Y = alt.Y("debt:Q", title="$ trillion", scale=YS,
          axis=alt.Axis(values=[38, 39, 40, 41], format="d", titleAngle=0, titleAlign="left", titleY=-10))   # horizontal, top left

hist = alt.Chart(debt).mark_line(strokeWidth=1.8).encode(X, Y, color=col)
limit = alt.Chart(pd.DataFrame({"debt": [LIMIT], "series": ["Limit"]})).mark_rule(strokeWidth=1.8).encode(Y, color=col)
election = alt.Chart(pd.DataFrame({"date": [ELECTION]})).mark_rule(color=NEUTRAL, strokeDash=[4, 3], strokeWidth=1.5).encode(alt.X("date:T", scale=XS))

def text(x, y, s, align="left", dx=0, dy=0, series=None):
    dd = pd.DataFrame({"date": [pd.Timestamp(x)], "debt": [y], "series": [series or "Limit"]})
    m = alt.Chart(dd).mark_text(text=s, align=align, dx=dx, dy=dy, **({"fontSize": FS} if FS else {}))
    enc = dict(x=alt.X("date:T", scale=XS), y=alt.Y("debt:Q", scale=YS))
    return m.encode(**enc, color=col) if series else m.encode(**enc, color=alt.value(NEUTRAL))

labels = (text(END, LIMIT, "Debt limit", align="right", dy=-10, series="Limit")
          + text(ELECTION, 41.45, "Midterm election day", align="right", dx=-6))

chart = (election + limit + hist + labels).properties(width=700, height=380)   # title/subtitle live in the text
if FS: chart = chart.configure_axis(labelFontSize=FS, titleFontSize=FS)

styles.save(chart, path=".", name="debt_limit_midterms", svg=True)

          date      debt  series
250 2026-11-04  40.55023  Actual
